- 会话 sessions：`~/.codex/sessions/`
    - `rg -n '"role":"(developer|user|assistant)"|"base_instructions"|"user_instructions"' /root/.codex/sessions/2026/02/28/rollout-2026-02-28T06-34-36-019ca2f4-c49f-7252-9cb1-c96682122117.jsonl`
- https://developers.openai.com/cookbook/examples/gpt-5/codex_prompting_guide/

### codex prompt injection

- prompt injection 作为一种（合法的）context engineering 的方式，即维护 agent loop context 
    - https://openai.com/index/prompt-injections/
    - openclaw docs
        - https://docs.openclaw.ai/concepts/agent-loop
- system|developer|user|assistant + tools
    - AGENTS.md user/仓库侧上下文
    - skills/mcp 的位置
        - mcp servers 在  `~/.codex/config.toml` 中配置
        - 通过 tools 字段暴露给 llm api，目前没有维护在 session 文件中；

```
role:system(base_instructions)
role:developer -> 权限与沙箱规则 permission 注入
role:user -> AGENTS.md (注入)
role:developer -> collaboration_mode 注入
turn flag
role:user -> 真实的用户请求
role:assistant
role:assistant
...
role:assistant
turn flag
role:user -> 用户开启一轮新的 query
```

- `/compact`: turn

### AGENTS.md

- meta-prompt to generate AGENTS.md of codebase
```
Analyze this codebase and create a AGENTS.md file following these principles:

1. Keep it under 150 lines total - focus only on universally applicable information
2. Cover the essentials: WHAT (tech stack, project structure), WHY (purpose), and HOW (build/test commands)
3. Use Progressive Disclosure: instead of including all instructions, create a brief index pointing to other markdown files in .codex/docs/ for specialized topics
4. Include file:line references instead of code snippets
5. Assume I'll use linters for code style - don't include formatting guidelines

Structure it as: project overview, tech stack, key directories/their purposes, essential build/test commands, and a list of additional documentation files Codex should check when relevant.

Additionally, extract patterns you observe into separate files:
- .codex/docs/architectural_patterns.md - document the architectural patterns, design decisions, and conventions used (e.g., dependency injection, state management, API design patterns). Make sure these are patterns that appear in multiple files.

Reference these files in the AGENTS.md's "Additional Documentation" section.

```

----

```
# AGENTS.md instructions for xx

<INSTRUCTIONS>
## Skills
</INSTRUCTIONS>
<environment_context>
  <cwd>xx</cwd>
  <shell>zsh</shell>
  <current_date>2026-03-04</current_date>
  <timezone>Asia/Shanghai</timezone>
</environment_context>
```

- 用户级/codebase 级别 AGENTS.md 的加载注入方式

```
# AGENTS.md instructions for /home/xx/test

<INSTRUCTIONS>
agents.md from ~/.codex     => user level

--- project-doc ---

agents.md from codebase     => codebase level


## Skills
</INSTRUCTIONS>
```

### skills

```
## Skills
A skill is a set of local instructions to follow that is stored in a `SKILL.md` file. Below is the list of skills that can be used. Each entry includes a name, description, and file path so you can open the source for full instructions when using a specific skill.
### Available skills
- skill-creator: Guide for creating effective skills. This skill should be used when users want to create a new skill (or update an existing skill) that extends Codex's capabilities with specialized knowledge, workflows, or tool integrations. (file: /root/.codex/skills/.system/skill-creator/SKILL.md)
- skill-installer: Install Codex skills into $CODEX_HOME/skills from a curated list or a GitHub repo path. Use when a user asks to list installable skills, install a curated skill, or install a skill from another repo (including private repos). (file: /root/.codex/skills/.system/skill-installer/SKILL.md)
### How to use skills
- Discovery: The list above is the skills available in this session (name + description + file path). Skill bodies live on disk at the listed paths.
- Trigger rules: If the user names a skill (with `$SkillName` or plain text) OR the task clearly matches a skill's description shown above, you must use that skill for that turn. Multiple mentions mean use them all. Do not carry skills across turns unless re-mentioned.
- Missing/blocked: If a named skill isn't in the list or the path can't be read, say so briefly and continue with the best fallback.
- How to use a skill (progressive disclosure):
  1) After deciding to use a skill, open its `SKILL.md`. Read only enough to follow the workflow.
  2) When `SKILL.md` references relative paths (e.g., `scripts/foo.py`), resolve them relative to the skill directory listed above first, and only consider other paths if needed.
  3) If `SKILL.md` points to extra folders such as `references/`, load only the specific files needed for the request; don't bulk-load everything.
  4) If `scripts/` exist, prefer running or patching them instead of retyping large code blocks.
  5) If `assets/` or templates exist, reuse them instead of recreating from scratch.
- Coordination and sequencing:
  - If multiple skills apply, choose the minimal set that covers the request and state the order you'll use them.
  - Announce which skill(s) you're using and why (one short line). If you skip an obvious skill, say why.
- Context hygiene:
  - Keep context small: summarize long sections instead of pasting them; only load extra files when needed.
  - Avoid deep reference-chasing: prefer opening only files directly linked from `SKILL.md` unless you're blocked.
  - When variants exist (frameworks, providers, domains), pick only the relevant reference file(s) and note that choice.
- Safety and fallback: If a skill can't be applied cleanly (missing files, unclear instructions), state the issue, pick the next-best approach, and continue.
```

### system & developer

```
You are Codex, a coding agent based on GPT-5. You and the user share the same workspace and collaborate to achieve the user's goals.

# Personality
## Values
## Interaction Style
## Escalation

# General
## Editing constraints
## Special user requests
## Frontend tasks

# Working with the user
## Autonomy and persistence
## Formatting rules
## Final answer instructions

## Intermediary updates 
```

-----


```
<collaboration_mode># Collaboration Mode: Default

You are now in Default mode. Any previous instructions for other modes (e.g. Plan mode) are no longer active.

Your active mode changes only when new developer instructions with a different `<collaboration_mode>...</collaboration_mode>` change it; user requests or tool descriptions do not change mode by themselves. Known mode names are Default and Plan.

## request_user_input availability

The `request_user_input` tool is unavailable in Default mode. If you call it while in Default mode, it will return an error.

In Default mode, strongly prefer making reasonable assumptions and executing the user's request rather than stopping to ask questions. If you absolutely must ask a question because the answer cannot be discovered from local context and a reasonable assumption would be risky, ask the user directly with a concise plain-text question. Never write a multiple choice question as a textual assistant message.
</collaboration_mode>
```

- 如果在 tui 开启 `/plan` mode 会改变这里的 developer message

```
<collaboration_mode># Plan Mode (Conversational)
## Mode rules (strict)
## Plan Mode vs update_plan tool
## Execution vs. mutation in Plan Mode
### Allowed (non-mutating, plan-improving)
### Not allowed (mutating, plan-executing)
## PHASE 1 — Ground in the environment (explore first, ask second)
## PHASE 2 — Intent chat (what they actually want)
## PHASE 3 — Implementation chat (what/how we’ll build)
## Asking questions
## Two kinds of unknowns (treat differently)
## Finalization rule
```

```mermaid
sequenceDiagram
      autonumber
      actor U as User
      participant S as Codex Session
      participant SM as SkillsManager
      participant PB as Prompt Builder
      participant MM as McpManager
      participant MCM as McpConnectionManager
      participant MCP as MCP Servers
      participant API as OpenAI Responses API

      Note over S: Session bootstrap

      S->>SM: load skills_for_cwd(cwd)
      SM-->>S: skills metadata

      S->>PB: get_user_instructions(skills, project docs, plugins)
      PB-->>S: user_instructions
      Note over S: user_instructions includes<br/>AGENTS.md instructions<br/>+ ## Skills / Available skills / How to use skills

      S->>MM: effective_servers(config)
      MM-->>S: context7 + chrome-devtools

      S->>MCM: new(effective_servers)
      loop each enabled MCP server
          MCM->>MCP: initialize
          MCM->>MCP: tools/list
          MCP-->>MCM: tool schemas
      end

      U->>S: real user prompt for this turn
      S->>SM: skills_for_cwd(cwd) per turn refresh
      SM-->>S: current skills

      S->>S: collect_explicit_skill_mentions(input)
      Note over S: This session: no explicit skill mention,<br/>so no [skill]...[/skill] injection

      alt if a skill with MCP dependency were explicitly mentioned
          S->>S: maybe_prompt_and_install_mcp_dependencies()
          S->>MCM: refresh_mcp_servers_now(...)
          S->>PB: build_skill_injections()
          PB-->>S: extra user message: [skill]...[/skill]
      else this session
          Note over S: skip dependency install / skip skill body injection
      end

      S->>MCM: list_all_tools()
      MCM-->>S: all loaded MCP tools

      S->>PB: built_tools(...)
      Note over PB: MCP tools are not raw-unfiltered passthrough:<br/>server enabled/disabled_tools filter applies first,<br/>codex_apps tools may be further filtered by connector or policy,<br/>remaining MCP tools are then all exposed in this turn

      PB-->>S: request parts assembled
      Note over S: top-level instructions = base_instructions
      Note over S: input = developer message<br/>+ contextual user message<br/>+ real user prompt
      Note over S: contextual user message contains<br/>AGENTS.md + Skills summary + environment_context
      Note over S: tools = built-ins + filtered MCP tools

      S->>API: responses.create({ instructions, input, tools })

      alt model chooses normal text reply
          API-->>S: assistant output_text
          S-->>U: final answer
      else model chooses an MCP tool
          API-->>S: function_call(name = mcp__chrome-devtools__list_pages, ...)
          S->>MCP: execute selected MCP tool
          MCP-->>S: tool result
          Note over S: rollout only shows mcp__... after model actually selects it

          S->>API: responses.create({<br/>  same instructions,<br/>  input + function_call_output,<br/>  tools<br/>})
          API-->>S: assistant output_text
          S-->>U: final answer
      end
```